# 🏪 YOLO-World Local Batch Sign Detection

Detect and crop 'storefront signs' using YOLO-World locally and export results to CSV.


In [1]:
import os
import cv2
import torch
import pandas as pd
from mmengine.config import Config
from mmdet.apis import inference_detector, init_detector


ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Load YOLO-World model
config_path = 'configs/yoloworld/yoloworld_s.py'
checkpoint_path = 'checkpoints/yoloworld_s.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = init_detector(config_path, checkpoint_path, device=device)
print("Model loaded.")


In [ ]:
input_dir = "RICHMOND" # test with 1 county folder
output_dir = "RICHMOND_cropped"
os.makedirs(output_dir, exist_ok=True)

results = []
target_classes = ["storefront sign"]

for fname in os.listdir(input_dir):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    image_path = os.path.join(input_dir, fname)
    try:
        result = inference_detector(model, image_path, target_classes)
        bboxes = result.pred_instances.bboxes.cpu().numpy()
        scores = result.pred_instances.scores.cpu().numpy()

        img = cv2.imread(image_path)
        detected = False

        for i, (box, score) in enumerate(zip(bboxes, scores)):
            if score < 0.3:
                continue
            x1, y1, x2, y2 = map(int, box)
            cropped = img[y1:y2, x1:x2]
            if cropped.size > 0:
                crop_path = os.path.join(output_dir, f"{os.path.splitext(fname)[0]}_{i}.jpg")
                cv2.imwrite(crop_path, cropped)
                detected = True

        results.append({
            "name": fname,
            "has_sign": "yes" if detected else "no",
            "num_boxes": len(bboxes),
            "max_score": float(scores[0]) if len(scores) > 0 else 0
        })

    except Exception as e:
        results.append({"name": fname, "has_sign": "error", "num_boxes": 0, "max_score": 0})
        print(f"Error on {fname}: {e}")


In [ ]:
df = pd.DataFrame(results)
df.to_csv("yoloworld_local_results.csv", index=False)
df.head()
# I removed the file because is not working
